# Results: Aggregate, Analyze, Export

Run this **after all workers finish**. Nothing here reruns experiments.

### Sections
1. **Load & Check** — aggregate CSVs, verify completeness
2. **Overview** — quick pivot tables and rankings
3. **Publication Tables** — LaTeX with bold-best formatting
4. **Publication Figures** — Pareto fronts, gradient conflict, bar charts
5. **Paper Numbers** — extract specific values for inline references
6. **Custom Analysis** — scratchpad for ad-hoc queries during writing

---
## 0. Setup

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, HTML

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ----- Path setup -----
DRIVE_ROOT = '/content/drive/MyDrive/DecisionFocusedMTL'
MOSEK_LIC_PATH = f'{DRIVE_ROOT}/data/mosek.lic'

if os.path.exists(MOSEK_LIC_PATH):
    os.environ['MOSEKLM_LICENSE_FILE'] = MOSEK_LIC_PATH

for p in [DRIVE_ROOT, os.path.join(DRIVE_ROOT, 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)

%cd {DRIVE_ROOT}

# ----- Result directories -----
HC_DIR = os.path.join(DRIVE_ROOT, 'results', 'final', 'healthcare')
KN_DIR = os.path.join(DRIVE_ROOT, 'results', 'final', 'knapsack')
OUT_DIR = os.path.join(DRIVE_ROOT, 'results', 'final')
TABLES_DIR = os.path.join(OUT_DIR, 'tables')
FIGS_DIR = os.path.join(OUT_DIR, 'figures')

for d in [TABLES_DIR, FIGS_DIR]:
    os.makedirs(d, exist_ok=True)

# ----- Display settings -----
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 250)
pd.set_option('display.float_format', '{:.4f}'.format)

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 9, 'axes.labelsize': 9,
    'legend.fontsize': 7, 'figure.dpi': 150, 'savefig.dpi': 300,
    'savefig.bbox': 'tight', 'axes.grid': True, 'grid.alpha': 0.3,
})

print(f'Working directory: {os.getcwd()}')
print(f'Healthcare results: {HC_DIR}')
print(f'Knapsack results:   {KN_DIR}')
print('Setup OK')

---
## 1. Load & Check

In [ ]:
def load_all(results_dir):
    """Load all per-run CSVs from a results tree."""
    p = Path(results_dir)
    agg = p / 'stage_results_all.csv'
    if agg.exists():
        df = pd.read_csv(agg)
    else:
        csvs = sorted(p.rglob('stage_results.csv'))
        if not csvs:
            return pd.DataFrame()
        df = pd.concat([pd.read_csv(c) for c in csvs], ignore_index=True)
    # Normalize
    if 'lambda' in df.columns:
        df['lam'] = df['lambda'].fillna(0.0)
    else:
        df['lam'] = 0.0
    # Readable row labels: FPTO(lam=0)=PTO, FDFL-Scal(lam=0)=DFL
    def _label(r):
        m = r.get('method_label', r.get('method', '?'))
        l = r['lam']
        if m == 'FPTO' and l == 0: return 'PTO'
        if m == 'FDFL-Scal' and l == 0: return 'DFL'
        if m in ('FPTO', 'FDFL-Scal') and l > 0: return f'{m} (lam={l})'
        return m
    df['row_label'] = df.apply(_label, axis=1)
    return df

hc = load_all(HC_DIR)
kn = load_all(KN_DIR)

# Save aggregates back
for df, d in [(hc, HC_DIR), (kn, KN_DIR)]:
    if not df.empty:
        Path(d).mkdir(parents=True, exist_ok=True)
        df.to_csv(Path(d) / 'stage_results_all.csv', index=False)

print(f'Healthcare: {len(hc)} rows')
print(f'Knapsack:   {len(kn)} rows')

In [ ]:
# === Completeness check ===
EXPECTED = ['FPTO', 'SAA', 'WDRO', 'FDFL-Scal', 'FDFL-PCGrad', 'FDFL-MGDA', 'FDFL-CAGrad']
SEEDS = [11, 22, 33, 44, 55]

def check(df, name, extras):
    if df.empty:
        print(f'{name}: NO DATA'); return
    print(f'\n{name}: {len(df)} rows')
    if 'method_label' in df.columns:
        found = set(df['method_label'].unique())
        missing = set(EXPECTED) - found
        print(f'  Methods: {sorted(found)}' + (f'  MISSING: {missing}' if missing else ' (all present)'))
    if 'seed' in df.columns:
        s = sorted(df['seed'].unique())
        ok = s == SEEDS
        print(f'  Seeds: {s}' + (' (OK)' if ok else f' EXPECTED {SEEDS}'))
    for col, vals in extras:
        if col in df.columns:
            found_v = sorted(df[col].unique())
            print(f'  {col}: {found_v}')

check(hc, 'Healthcare', [('alpha_fair', [0.5,2.0]), ('hidden_dim', [64,128]), ('lam', [0,0.5,1,5])])
check(kn, 'Knapsack',   [('alpha_fair', [0.5,2.0]), ('unfairness_level', ['mild','medium','high']), ('lam', [0,0.5,1,5])])

---
## 2. Overview

In [ ]:
# --- Reusable constants ---
ROW_ORDER = [
    'PTO', 'FPTO (lam=0.5)', 'FPTO (lam=1.0)', 'FPTO (lam=5.0)',
    'SAA', 'WDRO',
    'DFL', 'FDFL-Scal (lam=0.5)', 'FDFL-Scal (lam=1.0)', 'FDFL-Scal (lam=5.0)',
    'FDFL-PCGrad', 'FDFL-MGDA', 'FDFL-CAGrad',
]

METRIC_NAMES = {
    'test_regret_normalized': 'Norm. Regret',
    'test_fairness': 'Pred. Fair. Viol.',
    'test_pred_mse': 'Pred. MSE',
    'avg_cos_dec_pred': 'cos(dec,pred)',
    'avg_cos_dec_fair': 'cos(dec,fair)',
    'avg_cos_pred_fair': 'cos(pred,fair)',
}

COLORS = {
    'PTO': '#636363', 'FPTO (lam=0.5)': '#1f77b4', 'FPTO (lam=1.0)': '#aec7e8',
    'FPTO (lam=5.0)': '#6baed6', 'SAA': '#e6550d', 'WDRO': '#756bb1',
    'DFL': '#c49c94', 'FDFL-Scal (lam=0.5)': '#ff7f0e',
    'FDFL-Scal (lam=1.0)': '#ffbb78', 'FDFL-Scal (lam=5.0)': '#fdd0a2',
    'FDFL-PCGrad': '#2ca02c', 'FDFL-MGDA': '#d62728', 'FDFL-CAGrad': '#9467bd',
}
MARKERS = {
    'PTO': 'o', 'SAA': 'D', 'WDRO': 'p', 'DFL': 's',
    'FDFL-PCGrad': '*', 'FDFL-MGDA': 'h', 'FDFL-CAGrad': 'd',
}
for l in ['0.5','1.0','5.0']:
    MARKERS[f'FPTO (lam={l})'] = '^'
    MARKERS[f'FDFL-Scal (lam={l})'] = 'v'

In [ ]:
# --- Helper functions ---

def pivot(df, metric, col_col='alpha_fair', row_order=ROW_ORDER):
    """Quick pivot: methods × conditions."""
    piv = df.pivot_table(index='row_label', columns=col_col, values=metric, aggfunc='mean')
    return piv.reindex([r for r in row_order if r in piv.index])

def mean_std(df, panel_col='alpha_fair', metrics=None, row_order=ROW_ORDER):
    """Publication-style mean +/- std table."""
    metrics = metrics or ['test_regret_normalized', 'test_fairness', 'test_pred_mse']
    valid = [m for m in metrics if m in df.columns]
    panels = sorted(df[panel_col].unique())
    grouped = df.groupby(['row_label', panel_col])
    means, stds = grouped[valid].mean(), grouped[valid].std()
    rows = []
    for method in (row_order or sorted(df['row_label'].unique())):
        if method not in df['row_label'].values: continue
        row = {'Method': method}
        for panel in panels:
            for m in valid:
                col = f'{METRIC_NAMES.get(m,m)}|{panel}'
                try:
                    row[col] = f'{means.loc[(method,panel),m]:.4f} +/- {stds.loc[(method,panel),m]:.4f}'
                except: row[col] = '--'
        rows.append(row)
    return pd.DataFrame(rows).set_index('Method')

def ranks(df, condition_cols=['alpha_fair'], metrics=None, row_order=ROW_ORDER):
    """Average rank per method (1=best)."""
    metrics = metrics or ['test_regret_normalized', 'test_fairness', 'test_pred_mse']
    valid = [m for m in metrics if m in df.columns]
    all_r = []
    for _, grp in df.groupby(condition_cols):
        mm = grp.groupby('row_label')[valid].mean()
        for m in valid:
            ranked = mm[m].rank(ascending=True)
            for method, rank in ranked.items():
                all_r.append({'method': method, 'metric': m, 'rank': rank})
    if not all_r: return pd.DataFrame()
    avg = pd.DataFrame(all_r).groupby(['method','metric'])['rank'].mean().unstack()
    avg.columns = [METRIC_NAMES.get(m,m) for m in avg.columns]
    avg['Overall'] = avg.mean(axis=1)
    ordered = [r for r in row_order if r in avg.index]
    return avg.reindex(ordered).sort_values('Overall')

def get_val(df, method, alpha, metric, lam=None):
    """Extract mean/std for a specific (method, alpha, metric). For inline paper refs."""
    mask = (df['alpha_fair'] == alpha) & (df['row_label'] == method)
    v = df.loc[mask, metric].dropna()
    return {'mean': v.mean(), 'std': v.std(), 'n': len(v)} if len(v) else None

def to_latex(mean, std, fmt='.3f', bold=False):
    s = f'${mean:{fmt}} \\pm {std:{fmt}}$'
    return f'\\textbf{{{s}}}' if bold else s

print('Helpers loaded: pivot(), mean_std(), ranks(), get_val(), to_latex()')

In [ ]:
# --- Healthcare overview ---
if not hc.empty:
    print('Healthcare: Norm. Regret (mean over seeds, best hd)')
    display(pivot(hc, 'test_regret_normalized'))
    print('\nHealthcare: Fairness Violation')
    display(pivot(hc, 'test_fairness'))

In [ ]:
# --- Knapsack overview ---
if not kn.empty and 'unfairness_level' in kn.columns:
    print('Knapsack: Norm. Regret by unfairness')
    display(pivot(kn, 'test_regret_normalized', col_col='unfairness_level'))

In [ ]:
# --- Rankings ---
if not hc.empty:
    print('Healthcare Rankings (1=best):')
    display(ranks(hc))
if not kn.empty:
    cond = ['alpha_fair'] + (['unfairness_level'] if 'unfairness_level' in kn.columns else [])
    print('\nKnapsack Rankings (1=best):')
    display(ranks(kn, condition_cols=cond))

---
## 3. Publication Tables (mean +/- std)

In [ ]:
# --- Table 1: Healthcare ---
if not hc.empty:
    print('Table 1: Healthcare')
    display(mean_std(hc, panel_col='alpha_fair'))

In [ ]:
# --- Table 2: Knapsack per alpha ---
if not kn.empty and 'unfairness_level' in kn.columns:
    for alpha in sorted(kn['alpha_fair'].unique()):
        print(f'\nTable 2 (alpha={alpha}): Knapsack')
        display(mean_std(kn[kn['alpha_fair']==alpha], panel_col='unfairness_level'))

In [ ]:
# --- Generate LaTeX .tex files ---
os.makedirs(TABLES_DIR, exist_ok=True)
!python experiments/generate_tables.py \
    --healthcare-dir "{HC_DIR}" \
    --knapsack-dir "{KN_DIR}" \
    --output-dir "{TABLES_DIR}"

In [ ]:
# --- Display LaTeX source ---
for f in sorted(Path(TABLES_DIR).glob('*.tex')):
    print(f'\n{"="*70}\n{f.name}\n{"="*70}')
    print(f.read_text())

---
## 4. Publication Figures

In [ ]:
FIGS_DIR = os.path.join(OUT_DIR, 'figures')
os.makedirs(FIGS_DIR, exist_ok=True)

In [ ]:
# --- Fig 1: Pareto front (Healthcare) ---
if not hc.empty:
    alphas = sorted(hc['alpha_fair'].unique())
    fig, axes = plt.subplots(1, len(alphas), figsize=(5.5*len(alphas), 4.5))
    if len(alphas)==1: axes=[axes]
    for ax, alpha in zip(axes, alphas):
        sub = hc[hc['alpha_fair']==alpha]
        means = sub.groupby('row_label')[['test_regret_normalized','test_fairness']].mean()
        for method in ROW_ORDER:
            if method not in means.index: continue
            x, y = means.loc[method, 'test_regret_normalized'], means.loc[method, 'test_fairness']
            if np.isnan(x) or np.isnan(y): continue
            ax.scatter(x, y, c=COLORS.get(method,'#333'), marker=MARKERS.get(method,'o'),
                       s=80, edgecolors='k', linewidths=0.5, label=method, zorder=5)
        ax.set_xlabel('Normalized Decision Regret')
        ax.set_ylabel('Prediction Fairness Violation')
        ax.set_title(f'alpha = {alpha}')
        ax.legend(fontsize=5.5, loc='best', ncol=2)
    plt.tight_layout()
    fig.savefig(os.path.join(FIGS_DIR, 'fig1_pareto.pdf'))
    plt.show()

In [ ]:
# --- Fig 2: Regret by unfairness level (Knapsack) ---
if not kn.empty and 'unfairness_level' in kn.columns:
    focus = ['PTO','FPTO (lam=1.0)','SAA','DFL','FDFL-Scal (lam=1.0)',
             'FDFL-PCGrad','FDFL-MGDA','FDFL-CAGrad']
    alphas = sorted(kn['alpha_fair'].unique())
    fig, axes = plt.subplots(1, len(alphas), figsize=(6*len(alphas), 4))
    if len(alphas)==1: axes=[axes]
    for ax, alpha in zip(axes, alphas):
        sub = kn[(kn['alpha_fair']==alpha) & kn['row_label'].isin(focus)]
        piv = sub.pivot_table(index='row_label', columns='unfairness_level',
                              values='test_regret_normalized', aggfunc='mean')
        piv = piv.reindex([m for m in focus if m in piv.index])
        cols = [c for c in ['mild','medium','high'] if c in piv.columns]
        piv[cols].plot(kind='bar', ax=ax, width=0.8)
        ax.set_title(f'alpha = {alpha}')
        ax.set_ylabel('Norm. Regret')
        ax.tick_params(axis='x', rotation=40)
        ax.legend(title='Unfairness')
    plt.tight_layout()
    fig.savefig(os.path.join(FIGS_DIR, 'fig2_regret_by_unfairness.pdf'))
    plt.show()

In [ ]:
# --- Fig 3: Gradient conflict heatmap (Knapsack MOO methods) ---
if not kn.empty and 'unfairness_level' in kn.columns:
    cos_cols = [c for c in ['avg_cos_dec_pred','avg_cos_dec_fair','avg_cos_pred_fair'] if c in kn.columns]
    moo = kn[kn['row_label'].isin(['FDFL-PCGrad','FDFL-MGDA','FDFL-CAGrad'])]
    if not moo.empty and cos_cols:
        fig, axes = plt.subplots(1, len(cos_cols), figsize=(4*len(cos_cols), 3.5))
        if len(cos_cols)==1: axes=[axes]
        for ax, col in zip(axes, cos_cols):
            piv = moo.pivot_table(index='row_label', columns='unfairness_level',
                                  values=col, aggfunc='mean')
            cols = [c for c in ['mild','medium','high'] if c in piv.columns]
            piv = piv[cols]
            im = ax.imshow(piv.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
            for i in range(piv.shape[0]):
                for j in range(piv.shape[1]):
                    v = piv.values[i,j]
                    if not np.isnan(v):
                        ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                                fontsize=8, color='white' if abs(v)>0.4 else 'black')
            ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols)
            ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index)
            ax.set_title(METRIC_NAMES.get(col, col))
        fig.colorbar(im, ax=axes, shrink=0.8, label='Cosine similarity')
        plt.tight_layout()
        fig.savefig(os.path.join(FIGS_DIR, 'fig3_gradient_conflict.pdf'))
        plt.show()

In [ ]:
# --- Also run the automated figure generator ---
!python experiments/generate_figures.py \
    --healthcare-dir "{HC_DIR}" \
    --knapsack-dir "{KN_DIR}" \
    --output-dir "{FIGS_DIR}"

---
## 5. Paper Numbers

Extract specific values for inline references. Modify these for whatever you need.

In [ ]:
# --- Key numbers ---
if not hc.empty:
    print('=== Healthcare Key Metrics ===')
    for method in ['PTO', 'FPTO (lam=1.0)', 'SAA', 'DFL', 'FDFL-Scal (lam=1.0)',
                   'FDFL-PCGrad', 'FDFL-MGDA', 'FDFL-CAGrad']:
        for alpha in [0.5, 2.0]:
            r = get_val(hc, method, alpha, 'test_regret_normalized')
            if r:
                print(f'  {method:25s} a={alpha}: {r["mean"]:.4f} +/- {r["std"]:.4f} (n={r["n"]})')

In [ ]:
# --- Improvement over PTO baseline ---
if not hc.empty:
    print('=== % Improvement over PTO (Healthcare) ===')
    for alpha in [0.5, 2.0]:
        base = get_val(hc, 'PTO', alpha, 'test_regret_normalized')
        if not base: continue
        print(f'\nalpha={alpha} (PTO baseline = {base["mean"]:.4f}):')
        for m in ['FPTO (lam=1.0)','FDFL-Scal (lam=1.0)','FDFL-PCGrad','FDFL-MGDA','FDFL-CAGrad']:
            r = get_val(hc, m, alpha, 'test_regret_normalized')
            if r:
                pct = (base['mean'] - r['mean']) / base['mean'] * 100
                print(f'  {m:25s}: {pct:+.1f}% regret ({r["mean"]:.4f})')

In [ ]:
# --- LaTeX copy-paste snippets ---
if not hc.empty:
    print('=== LaTeX snippets ===')
    for m in ['FDFL-PCGrad', 'FDFL-MGDA', 'FDFL-CAGrad']:
        for a in [0.5, 2.0]:
            r = get_val(hc, m, a, 'test_regret_normalized')
            if r:
                print(f'% {m}, alpha={a}:')
                print(f'  {to_latex(r["mean"], r["std"])}')

---
## 6. Custom Analysis (Scratchpad)

Use `hc` and `kn` DataFrames + helper functions. Examples below — modify freely.

In [ ]:
# Q: Which method wins at alpha=2.0, high unfairness?
# kn[(kn['alpha_fair']==2.0) & (kn['unfairness_level']=='high')].groupby('row_label')['test_regret_normalized'].mean().sort_values()

In [ ]:
# Q: MOO vs best scalarized across all conditions?
# moo = hc[hc['row_label'].isin(['FDFL-PCGrad','FDFL-MGDA','FDFL-CAGrad'])].groupby('alpha_fair')['test_regret_normalized'].mean()
# scal = hc[hc['method_label']=='FDFL-Scal'].groupby('alpha_fair')['test_regret_normalized'].min()
# pd.DataFrame({'MOO avg': moo, 'Best scal': scal, 'Diff %': (scal-moo)/scal*100})

In [ ]:
# Q: Does fairness violation increase with unfairness level?
# kn.groupby(['row_label','unfairness_level'])['test_fairness'].mean().unstack().reindex(ROW_ORDER)

In [ ]:
# Q: Compact table — only select methods for main paper, full table in appendix?
# compact = ['PTO','FPTO (lam=1.0)','SAA','DFL','FDFL-Scal (lam=1.0)','FDFL-PCGrad','FDFL-MGDA','FDFL-CAGrad']
# display(mean_std(hc[hc['row_label'].isin(compact)], row_order=compact))

In [ ]:
# Your query here


In [ ]:
# --- List all output files ---
print('\nAll output files:')
for d in [os.path.join(OUT_DIR,'tables'), os.path.join(OUT_DIR,'figures')]:
    if os.path.exists(d):
        for f in sorted(Path(d).rglob('*')):
            if f.is_file():
                print(f'  {f.relative_to(OUT_DIR)} ({f.stat().st_size/1024:.0f} KB)')